# EDA và Clean — Brain Tumor MRI (glioma · meningioma · notumor · pituitary)

**Dữ liệu gốc:** `data/brain_mri(raw_dataset)/{Training,Testing}/<lớp>/` (chỉ đọc, không bị sửa).

- **Phần A – EDA (mục 1–9):** gộp `Training` và `Testing` thành một bộ, khảo sát số lượng, kích thước, định dạng, độ sáng, ảnh augment sẵn, ảnh trùng và ảnh bất thường.
- **Phần B – Clean (mục 10–15):** loại ảnh augment sẵn, ảnh trùng, nhóm nhãn mâu thuẫn; chia lại 70 / 15 / 15 theo nhóm; cắt viền đen, đưa về 224×224 mức xám và ghi ra bộ sạch.

Normalize và augment làm trong file mô hình.

**Đầu ra**

- `data/eda_outputs/` — biểu đồ, bảng của phần EDA.
- `data/brain_mri(clean_dataset)/{train,valid,test}/<lớp>/` — bộ dữ liệu sạch.
- `data/clean_outputs/` — danh sách ảnh bị loại, manifest và số liệu của phần Clean.

# PHẦN A — EDA

## 1. Import thư viện và cấu hình

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import json
import shutil
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.model_selection import StratifiedGroupKFold

RAW_DIR = "data/brain_mri(raw_dataset)"
OUT = "data/eda_outputs"
os.makedirs(OUT, exist_ok=True)

SPLITS = ["Training", "Testing"]
CLASSES = sorted(os.listdir(os.path.join(RAW_DIR, "Training")))
VEC = 128
SAME_T = 0.99
GROUP_T = 0.95
SPLIT_T = 0.90
SEED = 42

COLORS = dict(zip(CLASSES, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]))
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})


def save(fig, name):
    fig.savefig(os.path.join(OUT, name), bbox_inches="tight")
    plt.show()


def show_grid(paths, titles, ncols=6, size=2.0, name=None, title=None):
    n = len(paths)
    if n == 0:
        print("Không có ảnh:", title)
        return
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * size, nrows * (size + 0.3)))
    for i, ax in enumerate(np.array(axes).reshape(-1)):
        ax.axis("off")
        if i < n:
            ax.imshow(Image.open(paths[i]).convert("L"), cmap="gray")
            ax.set_title(titles[i], fontsize=7)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    if name:
        save(fig, name)
    else:
        plt.show()

print("Các lớp:", CLASSES)

## 2. Quét toàn bộ ảnh

Đọc mỗi ảnh một lần, ghi lại: kích thước, chế độ màu, mã MD5 của pixel, độ sáng, độ tương phản, tỉ lệ vùng não (phần sáng hơn nền đen) và một vector 128×128 mức xám đã chuẩn hoá để tính corr ở mục 7.

In [ ]:
rows, vecs, vecs_flip, bad = [], [], [], []
for split in SPLITS:
    for label in CLASSES:
        folder = os.path.join(RAW_DIR, split, label)
        for fname in sorted(os.listdir(folder)):
            path = os.path.join(folder, fname)
            try:
                img = Image.open(path)
                img.load()
                mode, (w, h) = img.mode, img.size
                g = img.convert("L")
                a = np.asarray(g)
            except Exception as e:
                bad.append({"path": path, "error": str(e)[:100]})
                continue
            small = np.asarray(g.resize((VEC, VEC), Image.BILINEAR), dtype=np.float32)
            v = small.ravel() - small.mean()
            vecs.append(v / (np.linalg.norm(v) + 1e-6))
            vf = small[:, ::-1].ravel() - small.mean()
            vecs_flip.append(vf / (np.linalg.norm(vf) + 1e-6))
            rows.append({
                "split": split, "label": label, "file": fname, "path": path,
                "key": f"{split}/{label}/{fname}", "mode": mode, "width": w, "height": h,
                "kb": os.path.getsize(path) / 1024, "md5": hashlib.md5(a.tobytes()).hexdigest(),
                "brightness": float(a.mean()), "contrast": float(a.std()),
                "brain_frac": float((small > max(10, small.max() * 0.08)).mean()),
                "is_aug": "aug" in fname.lower(),
            })

df = pd.DataFrame(rows)
V = np.stack(vecs).astype(np.float32)
VF = np.stack(vecs_flip).astype(np.float32)
print(f"Đọc được {len(df):,} ảnh · lỗi {len(bad)} ảnh")
df.head()

## 3. Số lượng ảnh theo lớp và theo thư mục gốc

In [ ]:
counts = pd.crosstab(df["label"], df["split"]).reindex(columns=SPLITS)
counts["tổng"] = counts.sum(axis=1)
counts.loc["tổng"] = counts.sum()
display(counts)

aug_counts = pd.crosstab(df["label"], [df["split"], df["is_aug"]])
print("Số ảnh có chữ 'aug' trong tên file (True):")
display(aug_counts)

fig, ax = plt.subplots(figsize=(7, 3.2))
x = np.arange(len(CLASSES))
for k, (split, col) in enumerate(zip(SPLITS, ["#2a78d6", "#eb6834"])):
    vals = counts.loc[CLASSES, split].values
    bars = ax.bar(x + (k - 0.5) * 0.38, vals, width=0.36, color=col, label=split)
    ax.bar_label(bars, fontsize=8, padding=2)
ax.set_xticks(x, CLASSES); ax.set_ylabel("số ảnh"); ax.legend(); ax.grid(axis="x", visible=False)
ax.set_title("Số ảnh mỗi lớp theo thư mục gốc")
save(fig, "fig03_class_counts.png")

## 4. Kích thước ảnh và chế độ màu

In [ ]:
df["size"] = df["width"].astype(str) + "×" + df["height"].astype(str)
print("Số kích thước khác nhau:", df["size"].nunique())
display(df["size"].value_counts().head(8).rename("số ảnh").to_frame().T)
print("Chế độ màu theo lớp:")
display(pd.crosstab(df["label"], df["mode"]))
print("Kích thước trung vị theo lớp:")
display(df.groupby(["label", "split"])[["width", "height", "kb"]].median().round(1))
print("Tỉ lệ ảnh 512×512 theo lớp:")
display((df["size"] == "512×512").groupby(df["label"]).mean().round(3).rename("tỉ lệ 512×512").to_frame().T)

fig, ax = plt.subplots(figsize=(6.5, 5))
for c in CLASSES:
    s = df[df["label"] == c]
    ax.scatter(s["width"], s["height"], s=10, alpha=0.5, color=COLORS[c], label=c)
ax.set_xlabel("width"); ax.set_ylabel("height"); ax.legend(); ax.set_title("Kích thước ảnh theo lớp")
save(fig, "fig04_image_sizes.png")

> **Nhận xét**
>
> - 7.200 ảnh, 447 kích thước khác nhau; phần lớn 512×512 (5.014 ảnh). Chế độ màu lẫn lộn: RGB 4.129, L 3.067, RGBA 3, P 1, nhưng nội dung đều là ảnh xám, nên phần Clean chuyển hết về mức xám.
> - **Kích thước gắn với lớp:** 96% ảnh glioma và pituitary, 86% meningioma là 512×512, nhưng **chỉ 1% ảnh notumor** có kích thước này (trung vị khoảng 230×240). Ảnh notumor đến từ một nguồn khác. Nếu giữ nguyên, mô hình có thể nhận ra notumor nhờ kích thước và khung hình thay vì nội dung, nên phần Clean cắt viền và đưa mọi ảnh về cùng 224×224.

## 5. Ảnh mẫu mỗi lớp

In [ ]:
for c in CLASSES:
    s = df[(df["label"] == c) & ~df["is_aug"]].sample(12, random_state=SEED)
    show_grid(s["path"].tolist(), (s["split"] + "\n" + s["file"]).tolist(), ncols=6,
              name=f"fig05_samples_{c}.png", title=c)

## 6. Độ sáng, độ tương phản và tỉ lệ vùng não

In [ ]:
stats = df.groupby("label")[["brightness", "contrast", "brain_frac"]].agg(["mean", "median"]).round(3)
display(stats)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, col in zip(axes, ["brightness", "contrast", "brain_frac"]):
    bp = ax.boxplot([df[df["label"] == c][col] for c in CLASSES], patch_artist=True, widths=0.55)
    ax.set_xticks(range(1, len(CLASSES) + 1), CLASSES, rotation=15)
    for patch, c in zip(bp["boxes"], CLASSES):
        patch.set_facecolor(COLORS[c]); patch.set_alpha(0.75)
    ax.set_title(col); ax.grid(axis="x", visible=False)
plt.tight_layout()
save(fig, "fig06_intensity.png")

> **Nhận xét**
>
> - notumor sáng nhất (trung vị 57) và tương phản cao nhất (59); glioma tối nhất (33). Nhiều ảnh notumor là ảnh T2/FLAIR (dịch não tuỷ sáng trắng), còn ba lớp có khối u chủ yếu là T1 có thuốc cản quang.
> - Đây là một **đường tắt khác**: độ sáng và kiểu chụp tách các lớp phần nào mà không cần nhìn thấy khối u. Cắt viền và resize không xoá được khác biệt này; augmentation thay đổi độ sáng/tương phản trong file mô hình giúp giảm bớt. Nên kiểm tra bằng Grad-CAM xem mô hình nhìn vào đâu.

## 7. Ảnh augment sẵn (`aug`)

Mỗi ảnh `aug` được so với mọi ảnh gốc (không phải `aug`) ở 6 dạng: giữ nguyên, lật ngang, lật dọc, xoay 90°, 180°, 270°. Ảnh gốc có corr cao nhất được coi là ảnh nguồn của nó.

In [ ]:
def unit(img):
    a = np.asarray(img.convert("L").resize((VEC, VEC), Image.BILINEAR), dtype=np.float32).ravel()
    a = a - a.mean()
    return a / (np.linalg.norm(a) + 1e-6)

TRANSFORMS = {"giữ nguyên": lambda m: m, "lật ngang": ImageOps.mirror, "lật dọc": ImageOps.flip,
              "xoay 90": lambda m: m.rotate(90, expand=True), "xoay 180": lambda m: m.rotate(180),
              "xoay 270": lambda m: m.rotate(270, expand=True)}
aug = df[df["is_aug"]]
base_idx = np.where(~df["is_aug"].values)[0]
Vb = V[base_idx]
aug_rows = []
for r in aug.itertuples():
    img = Image.open(r.path)
    S = np.stack([unit(t(img)) for t in TRANSFORMS.values()]) @ Vb.T
    t_id, b_id = np.unravel_index(S.argmax(), S.shape)
    src = df.iloc[base_idx[b_id]]
    aug_rows.append({"aug_key": r.key, "aug_split": r.split, "src_key": src["key"], "src_split": src["split"],
                     "transform": list(TRANSFORMS)[t_id], "corr": round(float(S.max()), 3)})
aug_src = pd.DataFrame(aug_rows)
print("Số ảnh aug:", len(aug_src))
display(pd.crosstab(aug_src["aug_split"], aug_src["src_split"], margins=True, margins_name="tổng")
        .rename_axis(index="ảnh aug nằm ở", columns="ảnh nguồn nằm ở"))
display(pd.cut(aug_src["corr"], [0, 0.8, 0.9, 0.95, 0.99, 1.01]).value_counts().sort_index().rename("số ảnh aug").to_frame().T)

ex = aug_src.sort_values("corr", ascending=False).iloc[::max(1, len(aug_src) // 6)].head(6)
paths, titles = [], []
for r in ex.itertuples():
    paths += [df.loc[df["key"] == r.aug_key, "path"].iat[0], df.loc[df["key"] == r.src_key, "path"].iat[0]]
    titles += [f"{r.aug_key.split('/')[-1]}\n(aug)", f"{r.src_key.split('/')[-1]}\n{r.transform} · corr={r.corr}"]
show_grid(paths, titles, ncols=6, name="fig07_aug_pairs.png", title="Ảnh aug (trái) và ảnh nguồn tìm được (phải)")
aug_src.to_csv(os.path.join(OUT, "aug_sources.csv"), index=False)

> **Nhận xét**
>
> - 203 ảnh `aug` (100 ở Training, 103 ở Testing), **tất cả thuộc lớp meningioma**: tác giả đã tự augment để lớp này đủ 1.400 / 400 ảnh.
> - Ảnh nguồn tìm được cho thấy đây là bản xoay, lật, zoom, chỉnh sáng của ảnh meningioma có sẵn. **87 ảnh aug trong Testing có nguồn nằm ở Training** — mô hình được kiểm tra trên biến thể của chính ảnh nó đã học.
> - Chỉ 20 ảnh aug có corr ≥ 0,95 với nguồn; phần lớn 0,8–0,95 vì đã bị biến đổi, nên bước tìm ảnh trùng bằng corr **không bắt được** chúng. Phải loại theo tên file. Phần Clean loại cả 203 ảnh.

## 8. Ảnh trùng (corr mọi cặp, không tính ảnh aug)

Ảnh được đưa về 128×128 mức xám (bộ dữ liệu có hàng trăm kích thước khác nhau nên phải đưa về cùng kích thước). Hệ số corr được tính cho **mọi cặp**, kể cả khác lớp và khác thư mục gốc.

Bộ dữ liệu có rất nhiều **bản lật ngang** của cùng một ảnh, nên corr của mỗi cặp là giá trị lớn hơn giữa (A so với B) và (A so với B lật ngang). Cột `lật` cho biết giá trị đó đến từ bản lật.

| Mức corr | Ý nghĩa |
|---|---|
| ≥ `SAME_T` (0,99) | Cùng một ảnh bị lưu nhiều lần, chỉnh sáng hoặc lật ngang |
| `GROUP_T` (0,95) – 0,99 | Thường là lát cắt liền kề của cùng một bệnh nhân |
| `SPLIT_T` (0,90) – 0,95 | Phần lớn vẫn là cùng một bệnh nhân: cùng lát cắt bị lọc mạnh, khác kiểu chụp (T2/FLAIR), hoặc lát cắt cách xa hơn |
| dưới 0,90 | Ngày càng nhiều ảnh của người khác, chỉ giống nhau về góc chụp |

Các cặp từ 0,80 trở lên được lưu lại để so sánh các khoảng.

In [ ]:
idx = np.where(~df["is_aug"].values)[0]
Vn, VFn = V[idx], VF[idx]
pairs = []
for s in range(0, len(Vn), 500):
    C0 = Vn[s:s + 500] @ Vn.T
    C1 = Vn[s:s + 500] @ VFn.T
    C = np.maximum(C0, C1)
    ii, jj = np.where(C >= 0.80)
    for i, j in zip(ii + s, jj):
        if i < j:
            pairs.append((int(idx[i]), int(idx[j]), float(C[i - s, j]), bool(C1[i - s, j] > C0[i - s, j])))
pairs = pd.DataFrame(pairs, columns=["a", "b", "corr", "lật"])
for c in ["split", "label", "key"]:
    pairs[c + "_a"] = df[c].values[pairs["a"]]
    pairs[c + "_b"] = df[c].values[pairs["b"]]
pairs["cùng lớp"] = pairs["label_a"] == pairs["label_b"]
pairs["cùng thư mục"] = pairs["split_a"] == pairs["split_b"]
pairs["mức"] = pd.cut(pairs["corr"], [0.80, 0.85, SPLIT_T, GROUP_T, SAME_T, 1.01], right=False,
                      labels=["0,80–0,85", "0,85–0,90", "0,90–0,95", "0,95–0,99", "≥ 0,99"])
print(f"Cặp có corr ≥ 0,80: {len(pairs):,}")
display(pd.crosstab(pairs["mức"], [pairs["cùng lớp"], pairs["cùng thư mục"]], margins=True, margins_name="tổng"))
band = pairs.groupby("mức", observed=False).agg(so_cap=("corr", "size"), khac_lop=("cùng lớp", lambda s: int((~s).sum())))
band["% khác lớp"] = (band["khac_lop"] / band["so_cap"] * 100).round(1)
display(band.rename(columns={"so_cap": "số cặp", "khac_lop": "cặp khác lớp"}))

md5_dup = df[~df["is_aug"] & df.duplicated("md5", keep=False)]
print("Nhóm giống hệt từng pixel (MD5):", md5_dup["md5"].nunique(), "· số ảnh:", len(md5_dup))

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(pairs["corr"], bins=np.arange(0.80, 1.0001, 0.005), color="#2a78d6")
for t in [SPLIT_T, GROUP_T, SAME_T]:
    ax.axvline(t, color="#52514e", lw=1)
ax.set_yscale("log"); ax.set_xlabel("corr"); ax.set_ylabel("số cặp (log)")
ax.set_title("Phân phối corr của các cặp ảnh rất giống nhau")
save(fig, "fig08a_corr_hist.png")

In [ ]:
def union_groups(pair_df):
    parent = list(range(len(df)))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for a, b in zip(pair_df["a"], pair_df["b"]):
        parent[find(a)] = find(b)
    return np.array([find(i) for i in range(len(df))])

df["same_group"] = union_groups(pairs[pairs["corr"] >= SAME_T])
df["patient_group"] = union_groups(pairs[pairs["corr"] >= GROUP_T])
df["split_group"] = union_groups(pairs[pairs["corr"] >= SPLIT_T])
na = df[~df["is_aug"]]
ss = na["same_group"].map(na["same_group"].value_counts())
ps = na["patient_group"].map(na["patient_group"].value_counts())
print(f"Nhóm ảnh giống hệt (≥ {SAME_T}): {na.loc[ss > 1, 'same_group'].nunique()} nhóm, {int((ss > 1).sum())} ảnh,"
      f" ảnh thừa {int((ss > 1).sum()) - na.loc[ss > 1, 'same_group'].nunique()}")
print(f"Nhóm ảnh giống nhau (≥ {GROUP_T}): {na.loc[ps > 1, 'patient_group'].nunique()} nhóm, {int((ps > 1).sum())} ảnh")

tr_groups = set(na.loc[na["split"] == "Training", "patient_group"])
tr_split = set(na.loc[na["split"] == "Training", "split_group"])
te = na[na["split"] == "Testing"]
te_same = set(na.loc[na["split"] == "Training", "same_group"])
leak = pd.DataFrame({
    "ảnh test": te["label"].value_counts(),
    f"có bản giống hệt trong train (≥ {SAME_T})": te[te["same_group"].isin(te_same)]["label"].value_counts(),
    f"có ảnh giống trong train (≥ {GROUP_T})": te[te["patient_group"].isin(tr_groups)]["label"].value_counts(),
    f"có ảnh giống trong train (≥ {SPLIT_T})": te[te["split_group"].isin(tr_split)]["label"].value_counts(),
}).reindex(CLASSES).fillna(0).astype(int)
leak.loc["tổng"] = leak.sum()
display(leak)

nlab = na.groupby("patient_group")["label"].nunique()
conflict_groups = nlab[nlab > 1].index
conflict = na[na["patient_group"].isin(conflict_groups)].sort_values("patient_group")
print("Nhóm giống nhau nhưng khác lớp:", len(conflict_groups), "· số ảnh:", len(conflict))
display(conflict[["patient_group", "key"]])

In [ ]:
def show_pairs(p, title, name, n=6):
    paths, titles = [], []
    for r in p.head(n).itertuples():
        paths += [df["path"].iat[r.a], df["path"].iat[r.b]]
        titles += [r.key_a.replace("/", "\n", 1), f"{r.key_b}\ncorr={r.corr:.3f}".replace("/", "\n", 1)]
    show_grid(paths, titles, ncols=4, size=2.6, name=name, title=title)

show_pairs(pairs[~pairs["cùng lớp"] & (pairs["corr"] >= GROUP_T)].sort_values("corr", ascending=False),
           "Cặp giống nhau nhưng KHÁC LỚP", "fig08b_cross_label.png", 8)
show_pairs(pairs[pairs["cùng lớp"] & ~pairs["cùng thư mục"] & (pairs["corr"] >= SAME_T)].sample(8, random_state=SEED),
           "Cặp giống hệt nằm ở Training và Testing", "fig08c_train_test.png", 8)
show_pairs(pairs[(pairs["corr"] >= GROUP_T) & (pairs["corr"] < SAME_T)].sample(8, random_state=SEED),
           "Cặp có corr 0,95–0,99", "fig08d_band_095_099.png", 8)
for lo, hi, name in [(0.90, 0.95, "fig08e_band_090_095.png"), (0.85, 0.90, "fig08f_band_085_090.png"),
                     (0.80, 0.85, "fig08g_band_080_085.png")]:
    sub = pairs[(pairs["corr"] >= lo) & (pairs["corr"] < hi)]
    show_pairs(sub.sample(min(8, len(sub)), random_state=SEED), f"Cặp có corr {lo:.2f}–{hi:.2f} (lấy ngẫu nhiên)", name, 8)
pairs.to_csv(os.path.join(OUT, "similar_pairs.csv"), index=False)

> **Nhận xét**
>
> - Tính cả bản lật ngang, có **49.313 cặp corr ≥ 0,80**. Bộ dữ liệu chứa **rất nhiều bản lật ngang của cùng một ảnh** ở cả 4 lớp: trong 7.810 cặp ≥ 0,99 có 4.699 cặp (60%) chỉ lộ ra khi so với bản lật. Nếu chỉ so ảnh gốc thì sẽ bỏ sót phần lớn ảnh trùng.
> - Gom các cặp ≥ 0,99 được 2.180 nhóm gồm 5.326 ảnh, tức **3.146 ảnh thừa** (gần một nửa bộ dữ liệu). Có 277 nhóm giống hệt từng pixel (591 ảnh).
> - Xem bằng mắt từng khoảng (hình 8d–8g) và tỉ lệ cặp khác lớp:
>
>   | Khoảng corr | Số cặp | % khác lớp | Khi xem ảnh |
>   |---|---|---|---|
>   | ≥ 0,99 | 7.810 | 0,3% | Cùng một ảnh (lưu lại, chỉnh sáng hoặc lật ngang) |
>   | 0,95–0,99 | 4.851 | 0,0% | Lát cắt liền kề của cùng bệnh nhân |
>   | 0,90–0,95 | 5.220 | 1,3% | **Phần lớn vẫn cùng bệnh nhân**: cùng lát cắt bị lọc mạnh, khác kiểu chụp T2/FLAIR, hoặc lát cắt xa hơn |
>   | 0,85–0,90 | 6.919 | 13,8% | Khoảng một nửa cùng bệnh nhân, nửa còn lại là người khác có khối u ở vị trí giống nhau |
>   | 0,80–0,85 | 24.513 | 37,0% | Chủ yếu là người khác, chỉ cùng góc chụp |
>
>   Một bệnh nhân không thể mang hai nhãn khác nhau, nên tỉ lệ cặp khác lớp tăng là dấu hiệu corr bắt đầu nối ảnh của những người khác nhau.
> - **Rò rỉ giữa Training và Testing:** **1.062 / 1.497 ảnh test** (không tính aug) có bản giống hệt hoặc bản lật trong train; tính đến mức 0,90 là 1.229 ảnh (82%). Riêng notumor là 398 / 400. Accuracy đo trên split gốc vì vậy chủ yếu đo khả năng nhớ ảnh.
> - **9 nhóm (32 ảnh) giống nhau ở mức ≥ 0,95 nhưng mang hai nhãn khác nhau** (glioma / meningioma và meningioma / pituitary) — chắc chắn có nhãn sai, phần Clean loại cả nhóm.

## 8b. Chọn ngưỡng gom nhóm để chia tập

Gom nhóm ở ngưỡng càng thấp thì bắt được càng nhiều ảnh cùng bệnh nhân, nhưng các nhóm cũng nối dây chuyền với nhau (A giống B, B giống C…) thành nhóm rất lớn. Một nhóm phải nằm trọn trong một tập, nên nhóm quá lớn làm việc chia 70 / 15 / 15 lệch đi. Bảng dưới thử chia ở nhiều ngưỡng (không tính ảnh aug và ảnh giống hệt).

In [ ]:
base = df[~df["is_aug"]].drop_duplicates("same_group")
rows_t = []
for t in [0.95, 0.92, 0.90, 0.88, 0.85, 0.80]:
    g = pd.Series(union_groups(pairs[pairs["corr"] >= t]))[base.index].values
    vc = pd.Series(g).value_counts()
    fold_t = np.zeros(len(base), dtype=int)
    for k, (_, te_idx) in enumerate(StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=SEED)
                                    .split(base, base["label"], groups=g)):
        fold_t[te_idx] = k
    test_pct = pd.Series(fold_t < 3).groupby(base["label"].values).mean() * 100
    rows_t.append({"ngưỡng": t, "ảnh trong nhóm": int((pd.Series(g).map(vc) > 1).sum()),
                   "nhóm lớn nhất": int(vc.max()),
                   "% test thấp nhất": round(test_pct.min(), 1), "% test cao nhất": round(test_pct.max(), 1)})
display(pd.DataFrame(rows_t).set_index("ngưỡng"))

> **Nhận xét**
>
> - Từ 0,95 xuống 0,90, số ảnh nằm trong nhóm tăng từ 449 lên 1.776 nhưng nhóm lớn nhất chỉ 28 ảnh, và mọi lớp vẫn có khoảng 15% ảnh test.
> - Từ 0,85 trở xuống, các nhóm **nối dây chuyền** thành khối rất lớn (512 ảnh ở 0,85; 1.385 ảnh ở 0,80), tỉ lệ test của các lớp lệch tới 32–62%: không còn chia 70 / 15 / 15 được.
> - **Chọn `SPLIT_T = 0,90`:** bắt được gần hết các cặp cùng bệnh nhân rõ ràng mà vẫn chia cân bằng. Phần rò rỉ còn lại nằm ở khoảng 0,85–0,90, nơi khoảng một nửa các cặp là cùng bệnh nhân; đây là giới hạn không xử lý hết được khi không có mã bệnh nhân.

## 9. Ảnh bất thường

Xem bằng mắt các ảnh ở hai đầu: nhỏ nhất, tối nhất, sáng nhất, vùng não nhỏ nhất. Ảnh nào không phải ảnh MRI não hoặc chất lượng quá kém thì ghi vào `REMOVE_KEYS` ở mục 10.

In [ ]:
na = df[~df["is_aug"]].copy()
na["short_side"] = na[["width", "height"]].min(axis=1)
for col, asc, title, name in [("short_side", True, "12 ảnh nhỏ nhất", "fig09a_smallest.png"),
                              ("brightness", True, "12 ảnh tối nhất", "fig09b_darkest.png"),
                              ("brightness", False, "12 ảnh sáng nhất", "fig09c_brightest.png"),
                              ("brain_frac", True, "12 ảnh có vùng não nhỏ nhất", "fig09d_small_brain.png")]:
    s = na.sort_values(col, ascending=asc).head(12)
    show_grid(s["path"].tolist(), (s["label"] + "\n" + s["file"] + f"\n{col}=" + s[col].round(2).astype(str)).tolist(),
              ncols=6, name=name, title=title)
print("Ảnh có cạnh ngắn < 150 px:", int((na["short_side"] < 150).sum()))
display(na[na["short_side"] < 150]["label"].value_counts().rename("số ảnh").to_frame().T)

> **Nhận xét**
>
> - Không có ảnh nào cạnh ngắn dưới 150 px.
> - 12 ảnh tối nhất gần như đều là glioma, 12 ảnh sáng nhất gần như đều là notumor — khớp với mục 6.
> - Một số ảnh notumor có dải chữ của trang web ở mép trên/dưới (ví dụ `Te-no_279`, `Te-no_297`); ảnh có vùng não nhỏ nhất là meningioma chụp với viền đen rộng. Các ảnh này vẫn hợp lệ.
> - Các lưới ở mục này **không** cho thấy ảnh CT, nhưng khi xem danh sách kNN ở mục 16 thì phát hiện bộ dữ liệu có **29 ảnh CT** (không phải MRI), phần lớn ở lớp notumor. Các ảnh đó được ghi vào `REMOVE_KEYS` ở mục 10.

## Lưu kết quả EDA

In [ ]:
df.drop(columns=["path"]).to_csv(os.path.join(OUT, "per_image_stats.csv"), index=False)
eda_results = {
    "n_images": int(len(df)), "n_unreadable": len(bad),
    "counts": {s: {c: int(v) for c, v in df[df["split"] == s]["label"].value_counts().items()} for s in SPLITS},
    "n_aug": int(df["is_aug"].sum()),
    "aug_from_other_split": int((aug_src["aug_split"] != aug_src["src_split"]).sum()),
    "n_sizes": int(df["size"].nunique()),
    "modes": {k: int(v) for k, v in df["mode"].value_counts().items()},
    "pairs_ge_same_t": int((pairs["corr"] >= SAME_T).sum()),
    "pairs_ge_group_t": int((pairs["corr"] >= GROUP_T).sum()),
    "pairs_ge_split_t": int((pairs["corr"] >= SPLIT_T).sum()),
    "test_with_train_twin_095": {k: int(v) for k, v in leak.iloc[:, 2].items()},
    "test_with_train_twin_090": {k: int(v) for k, v in leak.iloc[:, 3].items()},
    "conflict_groups": int(len(conflict_groups)), "conflict_images": int(len(conflict)),
    "thresholds": {"same": SAME_T, "group": GROUP_T, "split": SPLIT_T, "vec_size": VEC},
}
with open(os.path.join(OUT, "eda_results.json"), "w", encoding="utf-8") as f:
    json.dump(eda_results, f, ensure_ascii=False, indent=2)
print(json.dumps(eda_results, ensure_ascii=False, indent=2))

# PHẦN B — CLEAN

## 10. Cấu hình làm sạch

- `REMOVE_KEYS`: ảnh loại thủ công, dạng `{"Training/glioma/Tr-gl_0001.jpg": "lý do"}` (cột `key`, hoặc `original_key` trong `suspect_labels.csv`). Sửa xong thì chạy lại từ mục 10. Danh sách hiện tại có 32 ảnh, chọn sau khi xem mục 16 (xem nhận xét ở cuối mục 16):
  - **29 ảnh CT** (không phải MRI), 21 trong số đó nằm ở lớp notumor;
  - **3 ảnh notumor có tổn thương ngấm thuốc thấy rõ** (`Tr-no_993`, `Tr-no_1132`, `Tr-no_498`).
- `CROP`: `True` thì cắt bỏ nền đen quanh não trước khi resize, để não chiếm cùng tỉ lệ khung hình ở mọi lớp.
- `IMG_SIZE`: kích thước ảnh sạch (vuông, mức xám).

In [ ]:
CLEAN_DIR = "data/brain_mri(clean_dataset)"
CLEAN_OUT = "data/clean_outputs"
os.makedirs(CLEAN_OUT, exist_ok=True)
NEW_SPLITS = ["train", "valid", "test"]
IMG_SIZE = 224
CROP = True
REMOVE_KEYS = {
    "Training/notumor/Tr-no_1274.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_813.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_1307.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_244.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_610.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_702.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_638.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_959.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_520.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_408.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_34.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_857.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_100.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_1242.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_537.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_407.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_719.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_462.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_700.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_90.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_174.jpg": "ảnh CT, không phải MRI",
    "Testing/notumor/Te-no_167.jpg": "ảnh CT, không phải MRI",
    "Testing/glioma/Te-gl_318.jpg": "ảnh CT, không phải MRI",
    "Testing/glioma/Te-gl_74.jpg": "ảnh CT, không phải MRI",
    "Testing/glioma/Te-gl_372.jpg": "ảnh CT, không phải MRI",
    "Testing/meningioma/Te-me_8.jpg": "ảnh CT, không phải MRI",
    "Testing/meningioma/Te-me_278.jpg": "ảnh CT, không phải MRI",
    "Testing/meningioma/Te-me_26.jpg": "ảnh CT, không phải MRI",
    "Testing/meningioma/Te-me_208.jpg": "ảnh CT, không phải MRI",
    "Training/notumor/Tr-no_993.jpg": "notumor nhưng thấy rõ tổn thương",
    "Training/notumor/Tr-no_1132.jpg": "notumor nhưng thấy rõ tổn thương",
    "Training/notumor/Tr-no_498.jpg": "notumor nhưng thấy rõ tổn thương",
}


def crop_brain(g):
    a = np.asarray(g)
    m = a > max(10, a.max() * 0.08)
    if m.mean() < 0.01:
        return g
    ys, xs = np.where(m)
    pad = int(0.02 * max(a.shape))
    y0, y1 = max(0, ys.min() - pad), min(a.shape[0], ys.max() + 1 + pad)
    x0, x1 = max(0, xs.min() - pad), min(a.shape[1], xs.max() + 1 + pad)
    return g.crop((x0, y0, x1, y1))


def to_square(g, size):
    w, h = g.size
    s = max(w, h)
    canvas = Image.new("L", (s, s), 0)
    canvas.paste(g, ((s - w) // 2, (s - h) // 2))
    return canvas.resize((size, size), Image.LANCZOS)


def prepare(path):
    g = Image.open(path).convert("L")
    if CROP:
        g = crop_brain(g)
    return to_square(g, IMG_SIZE)

## 11. Loại ảnh

Thứ tự: ảnh `aug` → nhóm khác lớp → loại thủ công (kèm mọi bản trùng ≥ 0,99 của ảnh đó, để không còn sót bản sao của ảnh đã loại) → ảnh giống hệt (mỗi nhóm ≥ 0,99 giữ một ảnh có độ phân giải lớn nhất).

Trước bước giữ 1 ảnh/nhóm, corr được tính lại trên ảnh **đã cắt viền**: hai bản của cùng một ảnh nhưng khác khung hình (ví dụ một bản có viền đen rộng hơn) chỉ lộ ra là giống hệt sau khi cắt viền.

In [ ]:
reason = pd.Series("", index=df.index, dtype=object)
reason[df["is_aug"]] = "ảnh augment sẵn"
reason[df["patient_group"].isin(conflict_groups) & (reason == "")] = "nhóm giống nhau khác lớp"
unknown = sorted(set(REMOVE_KEYS) - set(df["key"]))
if unknown:
    print("Không tìm thấy:", unknown)
manual = df["key"].isin(REMOVE_KEYS) & (reason == "")
reason[manual] = "loại thủ công: " + df.loc[manual, "key"].map(REMOVE_KEYS)
group_reason = dict(zip(df.loc[df["key"].isin(REMOVE_KEYS), "same_group"], df.loc[df["key"].isin(REMOVE_KEYS), "key"].map(REMOVE_KEYS)))
twins = df["same_group"].isin(group_reason) & (reason == "")
reason[twins] = "loại thủ công (bản trùng): " + df.loc[twins, "same_group"].map(group_reason)
print(f"Loại thủ công: {int(manual.sum())} ảnh trong REMOVE_KEYS và {int(twins.sum())} bản trùng (≥ {SAME_T}, kể cả bản lật) của chúng")

cand_idx = np.where(reason.values == "")[0]
prepared = [prepare(df["path"].iat[i]) for i in cand_idx]
Vc = np.stack([unit(g) for g in prepared])
Vcf = np.stack([unit(ImageOps.mirror(g)) for g in prepared])
del prepared
extra, extra90 = [], []
for s0 in range(0, len(Vc), 500):
    C = np.maximum(Vc[s0:s0 + 500] @ Vc.T, Vc[s0:s0 + 500] @ Vcf.T)
    ii, jj = np.where(C >= SPLIT_T)
    for i, j in zip(ii + s0, jj):
        if i < j:
            extra90.append((int(cand_idx[i]), int(cand_idx[j])))
            if C[i - s0, j] >= SAME_T:
                extra.append((int(cand_idx[i]), int(cand_idx[j])))


def merge_groups(groups, extra_pairs):
    parent = list(range(len(groups)))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x
    for i, g in enumerate(groups):
        parent[find(i)] = find(int(g))
    for a, b in extra_pairs:
        parent[find(a)] = find(b)
    return np.array([find(i) for i in range(len(groups))])

n_before = df.loc[cand_idx, "same_group"].nunique()
df["same_group"] = merge_groups(df["same_group"].values, extra)
df["patient_group"] = merge_groups(df["patient_group"].values, extra)
df["split_group"] = merge_groups(df["split_group"].values, extra90)
print(f"Tính lại trên ảnh đã cắt viền: {len(extra)} cặp corr ≥ {SAME_T};"
      f" số ảnh còn lại nếu giữ 1 ảnh/nhóm giảm từ {n_before} xuống {df.loc[cand_idx, 'same_group'].nunique()}")
nlab2 = df.iloc[cand_idx].groupby("patient_group")["label"].nunique()
reason[df["patient_group"].isin(nlab2[nlab2 > 1].index) & (reason == "")] = "nhóm giống nhau khác lớp"

cand = df[reason == ""].assign(pixels=lambda d: d["width"] * d["height"])
keep_idx = cand.sort_values(["same_group", "pixels", "kb"], ascending=[True, False, False]).drop_duplicates("same_group").index
reason[(reason == "") & ~df.index.isin(keep_idx)] = "ảnh giống hệt (giữ 1/nhóm)"

df["remove_reason"] = reason
removed = df[reason != ""]
kept = df[reason == ""].copy()
display(pd.crosstab(removed["label"], removed["remove_reason"], margins=True, margins_name="tổng"))
before_after = pd.DataFrame({"ban đầu": df["label"].value_counts(), "còn lại": kept["label"].value_counts()}).reindex(CLASSES)
before_after["loại"] = before_after["ban đầu"] - before_after["còn lại"]
display(before_after)

> **Nhận xét**
>
> - Loại 3.399 ảnh: 203 ảnh aug, 32 ảnh thuộc nhóm khác lớp, 32 ảnh trong `REMOVE_KEYS` cùng 20 bản trùng của chúng, và 3.112 ảnh giống hệt (giữ 1 ảnh/nhóm). Còn **3.801 ảnh**.
> - Tính lại corr sau khi cắt viền tìm thêm 9 ảnh thừa (cùng ảnh nhưng khác khung hình).
> - Còn lại: glioma 1.459, pituitary 958, meningioma 826, **notumor 558**. notumor mất nhiều nhất (1.242 / 1.800 ảnh) vì hơn hai phần ba lớp này là bản sao, bản lật và ảnh CT. Bộ sạch lệch lớp rõ: glioma gấp khoảng 2,6 lần notumor. File mô hình dùng `class_weight` và báo cáo F1 macro.

## 12. Chia lại 70 / 15 / 15 theo nhóm

Dùng `StratifiedGroupKFold` với 20 phần: 3 phần làm test, 3 phần làm valid, 14 phần làm train. Mỗi nhóm ảnh giống nhau (corr ≥ `SPLIT_T` = 0,90, tính trên cả ảnh gốc lẫn ảnh đã cắt viền) chỉ nằm trong **một** tập, còn tỉ lệ các lớp ở ba tập được giữ gần như nhau. Lý do chọn 0,90 ở mục 8 và 8b.

Vì các nhóm có kích thước khác nhau, mỗi seed cho một cách chia hơi khác; notebook thử 30 seed và giữ cách chia có tỉ lệ valid / test của các lớp gần 15% nhất.

In [ ]:
def make_split(seed):
    fold = np.zeros(len(kept), dtype=int)
    sgkf = StratifiedGroupKFold(n_splits=20, shuffle=True, random_state=seed)
    for k, (_, te_idx) in enumerate(sgkf.split(kept, kept["label"], groups=kept["split_group"])):
        fold[te_idx] = k
    sp = np.select([fold < 3, fold < 6], ["test", "valid"], "train")
    pct = pd.crosstab(kept["label"].values, sp, normalize="index") * 100
    return sp, float((pct[["valid", "test"]] - 15).abs().max().max())

trials = {seed: make_split(seed) for seed in range(SEED, SEED + 30)}
best_seed = min(trials, key=lambda k: trials[k][1])
kept["split_new"] = trials[best_seed][0]
print(f"Thử 30 seed, chọn seed {best_seed}: tỉ lệ valid/test của mọi lớp lệch tối đa {trials[best_seed][1]:.2f} điểm % so với 15%")

split_counts = pd.crosstab(kept["label"], kept["split_new"]).reindex(index=CLASSES, columns=NEW_SPLITS)
split_counts["tổng"] = split_counts.sum(axis=1)
split_counts.loc["tổng"] = split_counts.sum()
display(split_counts)
display((split_counts[NEW_SPLITS].div(split_counts["tổng"], axis=0) * 100).round(1))
gs = kept["split_group"].map(kept["split_group"].value_counts())
print(f"Ảnh nằm trong nhóm (≥ {SPLIT_T}): {int((gs > 1).sum())} · nhóm lớn nhất: {int(gs.max())} ảnh")
print("Nhóm nằm ở nhiều hơn 1 tập:", int((kept.groupby("split_group")["split_new"].nunique() > 1).sum()))

## 13. Cắt viền đen, đưa về 224×224 mức xám và ghi bộ sạch

Vùng não được xác định bằng ngưỡng độ sáng (lớn hơn 8% giá trị sáng nhất), cắt theo khung bao quanh vùng đó (chừa 2% lề), đặt vào khung vuông nền đen để không méo ảnh, rồi resize về `IMG_SIZE`.

In [ ]:
if os.path.exists(CLEAN_DIR):
    shutil.rmtree(CLEAN_DIR)
for s in NEW_SPLITS:
    for c in CLASSES:
        os.makedirs(os.path.join(CLEAN_DIR, s, c), exist_ok=True)

new_paths = []
for i, r in enumerate(kept.itertuples()):
    g = prepare(r.path)
    dst = os.path.join(CLEAN_DIR, r.split_new, r.label, f"{r.label}_{i:05d}.jpg")
    g.save(dst, quality=95)
    new_paths.append(dst)
kept["clean_path"] = new_paths
print("Đã ghi", len(kept), "ảnh vào", CLEAN_DIR)

s = kept.groupby("label").sample(3, random_state=SEED)
paths, titles = [], []
for r in s.itertuples():
    paths += [r.path, r.clean_path]
    titles += [f"{r.label}\ngốc {r.width}×{r.height}", f"sạch {IMG_SIZE}×{IMG_SIZE}"]
show_grid(paths, titles, ncols=6, name=None, title="Ảnh gốc và ảnh sau khi cắt viền, resize")

## 14. Kiểm tra bộ dữ liệu sạch

In [ ]:
on_disk = {s: {c: len(os.listdir(os.path.join(CLEAN_DIR, s, c))) for c in CLASSES} for s in NEW_SPLITS}
on_disk = pd.DataFrame(on_disk)
assert int(on_disk.values.sum()) == len(kept)
assert (on_disk > 0).all().all()

clean_imgs = [Image.open(p).convert("L") for p in kept["clean_path"]]
Vk = np.stack([unit(g) for g in clean_imgs])
Vkf = np.stack([unit(ImageOps.mirror(g)) for g in clean_imgs])
del clean_imgs
sk = kept["split_new"].values
n_same_left, n_split_cross = 0, 0
for s0 in range(0, len(Vk), 500):
    C = np.maximum(Vk[s0:s0 + 500] @ Vk.T, Vk[s0:s0 + 500] @ Vkf.T)
    ii, jj = np.where(C >= SPLIT_T)
    for i, j in zip(ii + s0, jj):
        if i < j:
            n_same_left += int(C[i - s0, j] >= SAME_T)
            n_split_cross += int(sk[i] != sk[j])

cand_keys = set(kept["key"])
kp = pairs[pairs["key_a"].isin(cand_keys) & pairs["key_b"].isin(cand_keys) & (pairs["corr"] >= SPLIT_T)]
new_split = dict(zip(kept["key"], kept["split_new"]))
n_orig_cross = int(sum(new_split[a] != new_split[b] for a, b in zip(kp["key_a"], kp["key_b"])))

check = pd.Series({
    "số ảnh trong bộ sạch": f"{len(kept):,}  {on_disk.sum().to_dict()}",
    "ảnh aug còn lại": int(kept["is_aug"].sum()),
    f"nhóm (≥ {SPLIT_T}) nằm ở nhiều tập": int((kept.groupby("split_group")["split_new"].nunique() > 1).sum()),
    f"cặp ảnh sạch có corr ≥ {SAME_T}": n_same_left,
    f"cặp ảnh gốc có corr ≥ {SPLIT_T} nằm ở hai tập": n_orig_cross,
    f"cặp ảnh sạch có corr ≥ {SPLIT_T} nằm ở hai tập": n_split_cross,
    "lớp có mặt ở mỗi tập": " / ".join(str(int((on_disk[s] > 0).sum())) for s in NEW_SPLITS),
    "số ảnh trong thư mục gốc (không đổi)": int(sum(len(fs) for _, _, fs in os.walk(RAW_DIR))),
}, name="kết quả")
display(check.to_frame())

> **Nhận xét**
>
> - Bộ sạch có **3.801 ảnh: train 2.672 / valid 564 / test 565** (70,3 / 14,8 / 14,9); mọi lớp lệch không quá 0,3 điểm % so với 70 / 15 / 15.
> - Mọi kiểm tra đều đạt: không còn ảnh aug, không còn cặp ảnh sạch nào có corr ≥ 0,99 (tính cả bản lật), **không còn cặp nào có corr ≥ 0,90 nằm ở hai tập khác nhau** (trên cả ảnh gốc lẫn ảnh đã cắt viền), cả 4 lớp có mặt ở mỗi tập, thư mục gốc không bị sửa.
> - Bộ dữ liệu chỉ còn khoảng một nửa so với ban đầu: phần bị bỏ là bản sao, không phải thông tin mới, nên mô hình không mất gì; nhưng accuracy sẽ thấp hơn hẳn các kết quả công bố trên split gốc.

## 15. Lưu kết quả làm sạch

In [ ]:
kept[["clean_path", "split_new", "label", "key", "split", "width", "height", "same_group", "patient_group", "split_group"]].rename(
    columns={"split": "original_split", "key": "original_key"}).to_csv(os.path.join(CLEAN_OUT, "clean_manifest.csv"), index=False)
removed[["key", "label", "remove_reason"]].to_csv(os.path.join(CLEAN_OUT, "removed_images.csv"), index=False)
clean_results = {
    "clean_dir": CLEAN_DIR, "classes": CLASSES, "img_size": IMG_SIZE, "crop": CROP, "split_seed": int(best_seed),
    "n_raw": int(len(df)), "n_clean": int(len(kept)),
    "n_removed": {k: int(v) for k, v in removed["remove_reason"].value_counts().items()},
    "counts": {s: {c: int(on_disk.loc[c, s]) for c in CLASSES} for s in NEW_SPLITS},
    "thresholds": {"same": SAME_T, "group": GROUP_T, "split": SPLIT_T},
    "manual_removed": REMOVE_KEYS,
}
with open(os.path.join(CLEAN_OUT, "clean_results.json"), "w", encoding="utf-8") as f:
    json.dump(clean_results, f, ensure_ascii=False, indent=2)
print(json.dumps({k: v for k, v in clean_results.items() if k != "classes"}, ensure_ascii=False, indent=2))

## 16. Ảnh nghi gắn nhãn sai (kNN trên đặc trưng MobileNetV2)

Ảnh sạch được biến thành vector 1.280 số bằng MobileNetV2 (trọng số ImageNet, không train). Với mỗi ảnh, lấy `KNN_K` ảnh gần nhất **thuộc nhóm bệnh nhân khác** (bỏ qua các ảnh cùng `split_group`, nếu không ảnh nào cũng có hàng xóm là chính lát cắt bên cạnh của nó). Nếu ít nhất `KNN_NEED` / `KNN_K` hàng xóm cùng thuộc **một lớp khác** thì ảnh bị đánh dấu nghi ngờ.

Mục này **không xoá ảnh nào**: danh sách chỉ để xem bằng mắt. Ảnh nào sai rõ ràng thì thêm `original_key` vào `REMOVE_KEYS` ở mục 10 rồi chạy lại từ mục 10. Lần đầu chạy cần mạng để Keras tải trọng số MobileNetV2 (khoảng 9 MB).

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

KNN_K = 5
KNN_NEED = 4

backbone = MobileNetV2(include_top=False, weights="imagenet", pooling="avg", input_shape=(IMG_SIZE, IMG_SIZE, 3))
clean_paths = kept["clean_path"].tolist()
emb = []
for s0 in range(0, len(clean_paths), 64):
    x = np.stack([np.repeat(np.asarray(Image.open(p).convert("L"), dtype=np.float32)[..., None], 3, axis=-1)
                  for p in clean_paths[s0:s0 + 64]])
    emb.append(backbone.predict(preprocess_input(x), verbose=0))
emb = np.concatenate(emb)
emb /= np.linalg.norm(emb, axis=1, keepdims=True)

y_knn = kept["label"].map({c: i for i, c in enumerate(CLASSES)}).values
grp = kept["split_group"].values
sim = emb @ emb.T
sim[grp[:, None] == grp[None, :]] = -1
nn = np.argsort(-sim, axis=1)[:, :KNN_K]
del sim


def knn_flags(labels):
    votes = np.stack([np.bincount(labels[r], minlength=len(CLASSES)) for r in nn])
    return votes, (votes.argmax(1) != labels) & (votes.max(1) >= KNN_NEED)


votes, flag = knn_flags(y_knn)
kept["knn_label"] = [CLASSES[k] for k in votes.argmax(1)]
kept["knn_own"] = votes[np.arange(len(y_knn)), y_knn] / KNN_K
print(f"Độ chính xác kNN (k={KNN_K}): {(kept['knn_label'] == kept['label']).mean():.3f}")
display((pd.crosstab(kept["label"], kept["knn_label"], normalize="index") * 100).round(1)
        .rename_axis(index="nhãn gắn", columns="nhãn theo kNN (%)"))

suspect = kept[flag].sort_values(["knn_own", "label"])
print(f"Ảnh nghi gắn nhãn sai: {len(suspect)} / {len(kept):,} ({len(suspect) / len(kept):.1%})")
display(pd.crosstab(suspect["label"], suspect["knn_label"], margins=True, margins_name="tổng")
        .rename_axis(index="nhãn gắn", columns="nhãn theo kNN"))
suspect[["key", "label", "knn_label", "knn_own", "split_new", "clean_path"]].rename(
    columns={"key": "original_key"}).to_csv(os.path.join(CLEAN_OUT, "suspect_labels.csv"), index=False)
print("Đã lưu danh sách vào", os.path.join(CLEAN_OUT, "suspect_labels.csv"))

In [ ]:
for c in CLASSES:
    s = suspect[suspect["label"] == c].head(18)
    show_grid(s["clean_path"].tolist(), (s["key"].str.split("/").str[-1] + "\n" + s["label"] + " → " + s["knn_label"]).tolist(),
              ncols=6, size=2.1, name=f"fig16_suspect_{c}.png", title=f"Ảnh gắn nhãn {c} bị kNN đánh dấu (tối đa 18 ảnh)")

### Ước lượng khả năng phát hiện của kNN

Không có danh sách nhãn đúng để so, nên ước lượng gián tiếp: **cố ý đổi nhãn ngẫu nhiên** 2%, 5%, 10% số ảnh rồi xem kNN tìm lại được bao nhiêu (lặp 5 lần mỗi mức).

- **recall:** tỉ lệ ảnh bị đổi nhãn mà kNN đánh dấu được.
- **precision:** trong các ảnh bị đánh dấu, tỉ lệ đúng là ảnh bị đổi nhãn.

In [ ]:
rng = np.random.default_rng(SEED)
sim_rows = []
for rate in [0.02, 0.05, 0.10]:
    for rep in range(5):
        noisy = y_knn.copy()
        idx = rng.choice(len(noisy), int(rate * len(noisy)), replace=False)
        noisy[idx] = (noisy[idx] + rng.integers(1, len(CLASSES), len(idx))) % len(CLASSES)
        _, f = knn_flags(noisy)
        hit = int(f[idx].sum())
        sim_rows.append({"tỉ lệ nhãn sai cài vào": f"{rate:.0%}", "số ảnh bị đổi": len(idx),
                         "recall": hit / len(idx), "precision": hit / max(int(f.sum()), 1), "số ảnh bị đánh dấu": int(f.sum())})
display(pd.DataFrame(sim_rows).groupby("tỉ lệ nhãn sai cài vào", sort=False).mean().round(2))

> **Nhận xét**
>
> - kNN trên đặc trưng ImageNet đoán đúng **85,1%** nhãn. pituitary dễ nhất (93,1%); meningioma khó nhất (67,9%), hay bị lẫn sang glioma (17,6%) và pituitary (12,0%).
> - **178 ảnh (4,7%) bị đánh dấu.** Phần lớn là meningioma ↔ glioma và meningioma → pituitary (u màng não vùng cạnh hố yên trông giống u tuyến yên).
> - **Khi xem danh sách kNN ở lần chạy trước (trước khi loại), nhóm phát hiện được:**
>   - **29 ảnh CT** (xương sọ trắng đặc, mô não xám đồng nhất), không phải MRI: 22 ảnh notumor, 4 meningioma, 3 glioma. Tìm thêm bằng cách lấy các ảnh gần nhất với những ảnh CT đã thấy. Ảnh CT ở notumor không bị kNN đánh dấu vì hàng xóm của chúng cũng là CT notumor — tức là mô hình có thể nhận ra "notumor" chỉ vì ảnh là CT.
>   - **3 ảnh notumor có tổn thương ngấm thuốc thấy rõ:** `Tr-no_993` (tổn thương dạng vòng có phù xung quanh), `Tr-no_1132` và `Tr-no_498` (hai nốt ngấm thuốc; hai ảnh này là bản lật của nhau).
>
>   Cả 32 ảnh đã nằm trong `REMOVE_KEYS`, và 20 bản trùng của chúng cũng bị loại theo.
> - Các ảnh còn lại trong danh sách (glioma ↔ meningioma, khối vùng hố yên mang nhãn meningioma hoặc glioma như `Tr-gl_1356`…) cần chuyên môn chẩn đoán hình ảnh để kết luận; nhóm **không tự sửa**.
> - **Phép thử đổi nhãn giả:** kNN tìm lại được **67–72%** số ảnh bị đổi nhãn, nhưng khi chỉ có 2% nhãn sai thì chỉ **25%** số ảnh bị đánh dấu là sai thật. Danh sách chỉ dùng để thu hẹp phạm vi xem lại, không xoá tự động.

## Tổng kết

| Vấn đề | Bằng chứng | Xử lý |
|---|---|---|
| Ảnh augment sẵn | 203 ảnh `aug` của meningioma; 87 ảnh aug ở Testing tạo từ ảnh Training | Loại theo tên file |
| Ảnh giống hệt, bản lật ngang | 3.146 ảnh thừa (≥ 0,99, tính cả bản lật), 60% các cặp chỉ lộ ra khi so bản lật; 1.062 / 1.497 ảnh test có bản giống hệt trong train | Giữ 1 ảnh/nhóm |
| Ảnh cùng bệnh nhân | 4.851 cặp 0,95–0,99 và 5.220 cặp 0,90–0,95; 1.229 ảnh test có ảnh cùng bệnh nhân trong train | Giữ lại, gom nhóm ở corr ≥ 0,90 rồi chia theo nhóm (`StratifiedGroupKFold`) |
| Nhãn mâu thuẫn | 9 nhóm, 32 ảnh (glioma / meningioma, meningioma / pituitary) | Loại cả nhóm |
| Ảnh CT lẫn vào bộ MRI | 29 ảnh, 22 ở notumor | Loại thủ công cùng bản trùng (`REMOVE_KEYS`) |
| notumor có tổn thương rõ | 3 ảnh | Loại thủ công cùng bản trùng |
| Kích thước, khung hình gắn với lớp | 1% ảnh notumor là 512×512 so với 86–96% ở lớp khác | Cắt viền đen, đưa về 224×224 mức xám |
| Độ sáng, kiểu chụp gắn với lớp | notumor sáng nhất, glioma tối nhất | Augment độ sáng/tương phản trong file mô hình, kiểm tra bằng Grad-CAM |
| Nhãn sai không có bản trùng | kNN đánh dấu 178 ảnh (4,7%); phép thử đổi nhãn giả cho thấy chỉ khoảng 1/4 là sai thật ở mức nhiễu 2% | Chỉ xem lại, không xoá tự động (`REMOVE_KEYS`) |
| Lệch lớp sau làm sạch | notumor 558, meningioma 826, pituitary 958, glioma 1.459 | `class_weight`, báo cáo F1 macro |

**Giới hạn:** bộ dữ liệu không có mã bệnh nhân, nên ảnh cùng một người có corr dưới 0,90 (khoảng một nửa các cặp 0,85–0,90) vẫn có thể nằm ở hai tập khác nhau. Nhãn không được kiểm định lại; đây là bài tập phân loại ảnh, không phải công cụ chẩn đoán.